# Machine Learning & AI for Consumer Insurance
## Study Notes

**Topics covered:** Supervised Learning · Ensemble Methods · Unsupervised Learning · Deep Learning · Time Series · Model Evaluation  
**Domain context:** All examples use consumer insurance scenarios (auto, home, life)

---

| # | Algorithm | Type |
|---|-----------|------|
| 01 | Linear Regression | Supervised — Regression |
| 02 | Logistic Regression | Supervised — Classification |
| 03 | Decision Trees | Supervised — Classification & Regression |
| 04 | Random Forests | Ensemble — Bagging |
| 05 | Gradient Boosting | Ensemble — Boosting |
| 06 | Support Vector Machines | Supervised — Classification |
| 07 | K-Nearest Neighbors | Instance-Based Learning |
| 08 | Naive Bayes | Probabilistic — Classification |
| 09 | Neural Networks (MLP) | Deep Learning |
| 10 | K-Means Clustering | Unsupervised |
| 11 | DBSCAN | Unsupervised — Density-Based |
| 12 | PCA | Dimensionality Reduction |
| 13 | Time Series (SARIMA) | Temporal Forecasting |
| 14 | Model Evaluation | Metrics, Cross-Validation, Calibration |


In [ ]:
# ─── Imports (run this first) ─────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay, average_precision_score
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("All libraries loaded.")


---
## Module 01 — Linear Regression

**Definition:** Finds the best-fit line (or hyperplane) through data by learning a weight for each feature. The prediction is a weighted sum of inputs.

**Formula:**  
`ŷ = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ`

- `β₀` = intercept (baseline when all features are zero)  
- `βᵢ` = coefficient for feature `xᵢ` (how much that feature shifts the prediction)

**Training objective:** Minimise Mean Squared Error (MSE) — the average squared difference between predicted and actual values.  
`MSE = (1/n) Σ (yᵢ - ŷᵢ)²`

**Regularisation variants:**
- **Ridge (L2):** Adds penalty `λ Σ βᵢ²` — shrinks all coefficients, handles correlated features
- **Lasso (L1):** Adds penalty `λ Σ |βᵢ|` — drives some coefficients to exactly zero (automatic feature selection)

**Analogy:** An experienced actuary's pricing rule — *"add £200 per prior claim, subtract £30 per NCB year."* Linear regression learns those exact numbers automatically from thousands of historical policies.

**Insurance use cases:**
- Predict annual premium from risk features (baseline pricing model)
- Predict claim cost / severity
- Predict portfolio loss ratio by segment

---

**Scenario:** We have a portfolio of 1,000 auto policies. Each row has driver features and their actual annual premium. We want to learn which features drive premium up or down, and by how much.


In [ ]:
# ─── Linear Regression: Premium Pricing ──────────────────────────────────
# Generate a small synthetic auto insurance dataset
np.random.seed(42)
n = 1000

driver_age     = np.random.randint(18, 75, n)
ncb_years      = np.random.randint(0, 10, n)
prior_claims   = np.random.choice([0,1,2,3], n, p=[0.65, 0.22, 0.09, 0.04])
vehicle_group  = np.random.randint(1, 20, n)
postcode_risk  = np.random.uniform(0, 1, n)

# True premium = actuarial formula + noise
annual_premium = (
    200
    + 400 * postcode_risk
    + 200 * prior_claims
    - 30  * ncb_years
    + 5   * vehicle_group
    + 100 * (driver_age < 25).astype(int)
    + np.random.normal(0, 50, n)
).clip(150, 3000)

df = pd.DataFrame({
    'driver_age':    driver_age,
    'ncb_years':     ncb_years,
    'prior_claims':  prior_claims,
    'vehicle_group': vehicle_group,
    'postcode_risk': postcode_risk,
    'annual_premium': annual_premium
})

features = ['driver_age', 'ncb_years', 'prior_claims', 'vehicle_group', 'postcode_risk']
X = df[features]
y = df['annual_premium']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features (important for Ridge/Lasso so coefficients are comparable)
scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

# ─── Fit three variants ───────────────────────────────────────────────────
ols   = LinearRegression().fit(X_train_sc, y_train)
ridge = Ridge(alpha=10).fit(X_train_sc, y_train)
lasso = Lasso(alpha=5).fit(X_train_sc, y_train)

# ─── Evaluate ─────────────────────────────────────────────────────────────
print(f"{'Model':<18} {'MAE (£)':>10} {'RMSE (£)':>10} {'R²':>8}")
print("-" * 50)
for name, model in [('OLS', ols), ('Ridge (L2)', ridge), ('Lasso (L1)', lasso)]:
    pred = model.predict(X_test_sc)
    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2   = r2_score(y_test, pred)
    print(f"{name:<18} {mae:>10.2f} {rmse:>10.2f} {r2:>8.4f}")


In [ ]:
# ─── Coefficient interpretation ───────────────────────────────────────────
# Standardised coefficients: how much premium changes per std dev of each feature
# Positive = increases premium | Negative = decreases premium

coef_df = pd.DataFrame({
    'feature':     features,
    'coefficient': ols.coef_
}).sort_values('coefficient', ascending=False)

print("OLS Coefficients (standardised)")
print("Interpretation: £ change in premium per 1 std dev increase in feature")
print("-" * 55)
for _, row in coef_df.iterrows():
    direction = "↑  raises" if row['coefficient'] > 0 else "↓  lowers"
    print(f"  {row['feature']:<18}  {direction} premium by £{abs(row['coefficient']):.2f}")

print(f"
  Intercept (baseline premium): £{ols.intercept_:.2f}")


In [ ]:
# ─── Predicted vs Actual + Residuals ─────────────────────────────────────
pred_ols = ols.predict(X_test_sc)
residuals = y_test.values - pred_ols

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Predicted vs Actual — points should cluster around the diagonal
axes[0].scatter(y_test, pred_ols, alpha=0.4, s=18, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Premium (£)'); axes[0].set_ylabel('Predicted Premium (£)')
axes[0].set_title(f'Predicted vs Actual
R² = {r2_score(y_test, pred_ols):.4f}')
axes[0].legend()

# Residuals vs Fitted — should be random scatter around 0 (no pattern = good)
axes[1].scatter(pred_ols, residuals, alpha=0.4, s=18, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Fitted Values (£)'); axes[1].set_ylabel('Residuals (£)')
axes[1].set_title('Residuals vs Fitted
Random scatter = model assumptions met')

plt.suptitle('Module 01 — Linear Regression', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Mean residual: £{residuals.mean():.4f}  (should be ~0)")
print(f"Std residual:  £{residuals.std():.2f}")


---
## Module 02 — Logistic Regression

**Definition:** A *classification* algorithm (despite the name) that predicts the probability a data point belongs to a class. It applies the sigmoid function to a linear combination of features, squashing any real number into [0, 1].

**Sigmoid function:**  
`P(y=1 | X) = 1 / (1 + e^(−(β₀ + β₁x₁ + ... + βₙxₙ)))`

**Odds Ratios:** Exponentiating a coefficient gives its odds ratio:  
`Odds Ratio for xᵢ = e^βᵢ`
- OR > 1 → feature increases probability of the event  
- OR < 1 → feature decreases probability  
- OR = 1 → no effect  

**Class imbalance:** In insurance, fraud is rare (~3%). A model that always predicts "not fraud" gets 97% accuracy but catches nothing. Fix: use `class_weight='balanced'` and evaluate with ROC-AUC, not accuracy.

**Analogy:** Your fraud team's gut feeling — *"solicitor involved + late reporting + round claim amount = suspicious."* Logistic regression formalises that feeling, assigning precise weights to each warning sign and outputting a probability like 0.87.

**Insurance use cases:**
- Fraud detection — P(claim is fraudulent)
- Churn prediction — P(customer cancels at renewal)
- Underwriting — P(policy is acceptable risk)

---

**Scenario:** We have 800 claims. Some are fraudulent (~5%). We want to score each claim with a fraud probability, then route high-scoring claims to an investigation queue.


In [ ]:
# ─── Logistic Regression: Fraud Detection ────────────────────────────────
np.random.seed(42)
n = 800

prior_claims      = np.random.choice([0,1,2,3], n, p=[0.60, 0.25, 0.10, 0.05])
policy_age_months = np.random.randint(1, 120, n)
postcode_risk     = np.random.uniform(0, 1, n)
claim_amount      = np.random.lognormal(7.5, 0.8, n).clip(200, 50000)
vehicle_group     = np.random.randint(1, 20, n)

# Fraud probability driven by real risk factors
fraud_log_odds = (
    -3.0
    + 1.2 * (prior_claims > 2).astype(float)   # many prior claims = suspicious
    + 0.8 * postcode_risk                        # high-risk postcode
    + 0.6 * (policy_age_months < 6).astype(float)  # very new policy
)
fraud_prob = 1 / (1 + np.exp(-fraud_log_odds))
is_fraud   = (np.random.uniform(0, 1, n) < fraud_prob).astype(int)

fraud_df = pd.DataFrame({
    'prior_claims':      prior_claims,
    'policy_age_months': policy_age_months,
    'postcode_risk':     postcode_risk,
    'claim_amount':      claim_amount,
    'vehicle_group':     vehicle_group,
    'is_fraud':          is_fraud
})

print(f"Total claims:  {n}")
print(f"Fraud cases:   {is_fraud.sum()}  ({is_fraud.mean():.1%})")
print(f"Legit cases:   {(is_fraud==0).sum()}  ({(is_fraud==0).mean():.1%})")
print()
print("Class imbalance present — will use class_weight='balanced'")


In [ ]:
# ─── Train and evaluate ───────────────────────────────────────────────────
fraud_features = ['prior_claims', 'policy_age_months', 'postcode_risk',
                  'claim_amount', 'vehicle_group']

X_f = fraud_df[fraud_features]
y_f = fraud_df['is_fraud']

X_tr, X_te, y_tr, y_te = train_test_split(X_f, y_f, test_size=0.2, random_state=42, stratify=y_f)

fraud_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(
        class_weight='balanced',  # compensates for fraud being rare
        C=0.5,                    # regularisation (lower = stronger penalty on large coefficients)
        max_iter=1000
    ))
])
fraud_pipe.fit(X_tr, y_tr)

fraud_proba = fraud_pipe.predict_proba(X_te)[:, 1]   # P(fraud) for each claim
fraud_pred  = fraud_pipe.predict(X_te)

print(f"ROC-AUC:  {roc_auc_score(y_te, fraud_proba):.4f}")
print(f"Gini:     {2 * roc_auc_score(y_te, fraud_proba) - 1:.4f}  (standard insurance metric)")
print()
print(classification_report(y_te, fraud_pred, target_names=['Legitimate', 'Fraud']))


In [ ]:
# ─── Odds ratios: what actually drives fraud? ─────────────────────────────
lr_model    = fraud_pipe['model']
odds_ratios = np.exp(lr_model.coef_[0])

or_df = pd.DataFrame({
    'feature':     fraud_features,
    'coefficient': lr_model.coef_[0],
    'odds_ratio':  odds_ratios
}).sort_values('odds_ratio', ascending=False)

print("Odds Ratios:")
print("  OR > 1 → increases fraud probability | OR < 1 → protective")
print("-" * 62)
for _, row in or_df.iterrows():
    flag = "⚠ raises" if row['odds_ratio'] > 1 else "✓ lowers"
    print(f"  {row['feature']:<22}  OR = {row['odds_ratio']:.3f}  →  {flag} fraud odds")


In [ ]:
# ─── Claim routing by fraud score ─────────────────────────────────────────
# In production: use probability scores to route claims to queues.
# Binary 0/1 predictions waste the information in the probability.

print("Claim Routing by Fraud Probability Score:")
print("-" * 58)
print(f"  {'Queue':<22} {'Claims':>7}  {'Share':>7}  {'Actual fraud %':>15}")
print("-" * 58)

queues = {
    '🟢 Auto-Pay':       (0.00, 0.25),
    '🟡 Standard Review': (0.25, 0.60),
    '🔴 Investigate':     (0.60, 1.00),
}
for queue, (lo, hi) in queues.items():
    mask       = (fraud_proba >= lo) & (fraud_proba < hi)
    count      = mask.sum()
    fraud_rate = y_te[mask].mean() if count > 0 else 0
    print(f"  {queue:<22} {count:>7}  {count/len(y_te):>7.1%}  {fraud_rate:>15.1%}")

print()
print("The 'Investigate' queue should have a much higher fraud rate than the overall baseline.")


---
## Module 03 — Decision Trees

**Definition:** A decision tree learns a sequence of yes/no questions about features. At each node, it picks the question that best separates the target variable. The result is a flowchart of rules that anyone can read.

**Splitting criterion — Gini Impurity:**  
`Gini(S) = 1 − Σ pₖ²`  
where `pₖ` is the proportion of class k in node S.
- Gini = 0 → perfectly pure (all same class) ✅
- Gini = 0.5 → maximally mixed (50/50) ❌

At each node the algorithm tries every split on every feature and picks the one that most reduces Gini impurity (= maximises Information Gain).

**Overfitting risk:** A deep tree memorises the training data. Control with `max_depth` and `min_samples_leaf`.

**Analogy:** An underwriter's mental checklist — *"Is the property in a flood zone? → Yes → Built before 1970? → Yes → Apply heavy loading."* A decision tree learns that exact checklist from historical decisions.

**Insurance use cases:**
- Claims triage — route to fast-track, review, or investigation
- Underwriting rules extraction — what rules does the book actually follow?
- Regulatory explainability — fully auditable decision path per claim

---

**Scenario:** We want to decide whether to fast-track or investigate a claim. We train a shallow decision tree on historical claim outcomes, then read off the learned rules.


In [ ]:
# ─── Decision Tree: Claims Triage ─────────────────────────────────────────
np.random.seed(42)
n = 600

prior_claims      = np.random.choice([0,1,2,3], n, p=[0.60, 0.25, 0.10, 0.05])
claim_amount      = np.random.lognormal(7.5, 0.8, n).clip(200, 50000)
policy_age_months = np.random.randint(1, 120, n)
postcode_risk     = np.random.uniform(0, 1, n)
ncb_years         = np.random.randint(0, 10, n)

fraud_log_odds = (
    -3.0
    + 1.5 * (prior_claims > 1).astype(float)
    + 0.8 * postcode_risk
    + 0.7 * (policy_age_months < 6).astype(float)
)
is_fraud = (np.random.uniform(0, 1, n) < 1/(1+np.exp(-fraud_log_odds))).astype(int)

triage_df = pd.DataFrame({
    'prior_claims':      prior_claims,
    'claim_amount':      claim_amount,
    'policy_age_months': policy_age_months,
    'postcode_risk':     postcode_risk,
    'ncb_years':         ncb_years,
    'is_fraud':          is_fraud
})

triage_features = ['prior_claims', 'claim_amount', 'policy_age_months', 'postcode_risk', 'ncb_years']
X_t = triage_df[triage_features]
y_t = triage_df['is_fraud']

X_tr, X_te, y_tr, y_te = train_test_split(X_t, y_t, test_size=0.2, random_state=42, stratify=y_t)

# ─── Shallow tree — interpretability is the goal here, not pure accuracy ──
tree = DecisionTreeClassifier(
    max_depth=4,            # keep it shallow so humans can read it
    min_samples_leaf=20,    # each leaf must represent at least 20 claims
    class_weight='balanced',
    criterion='gini',
    random_state=42
)
tree.fit(X_tr, y_tr)

tree_proba = tree.predict_proba(X_te)[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_te, tree_proba):.4f}")
print()
print("Learned triage rules (depth = 4):")
print("=" * 55)
print(export_text(tree, feature_names=triage_features))


In [ ]:
# ─── Gini impurity and the overfitting problem ────────────────────────────

# 1. Gini impurity as a function of class proportion
p = np.linspace(0, 1, 200)
gini = 2 * p * (1 - p)   # for binary classification: Gini = 1 - p² - (1-p)²

# 2. Depth vs performance (overfitting curve)
depths = range(1, 16)
train_auc, test_auc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, class_weight='balanced', random_state=42)
    t.fit(X_tr, y_tr)
    train_auc.append(roc_auc_score(y_tr, t.predict_proba(X_tr)[:,1]))
    test_auc.append(roc_auc_score(y_te, t.predict_proba(X_te)[:,1]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(p, gini, color='steelblue', linewidth=2.5)
axes[0].fill_between(p, gini, alpha=0.12, color='steelblue')
axes[0].axvline(0.5, color='crimson', linestyle='--', linewidth=1.5, label='Max impurity (0.5)')
axes[0].scatter([0, 1], [0, 0], color='seagreen', s=70, zorder=5, label='Pure nodes (Gini = 0)')
axes[0].set_xlabel('Proportion of Fraud in Node'); axes[0].set_ylabel('Gini Impurity')
axes[0].set_title('Gini Impurity
Lower = purer node = better split')
axes[0].legend()

axes[1].plot(depths, train_auc, 'o-', color='darkorange', linewidth=2, label='Train AUC')
axes[1].plot(depths, test_auc,  's-', color='steelblue',  linewidth=2, label='Test AUC')
axes[1].axvline(4, color='crimson', linestyle='--', linewidth=1.5, label='Our choice (depth=4)')
axes[1].fill_between(depths, train_auc, test_auc, alpha=0.12, color='crimson', label='Overfitting gap')
axes[1].set_xlabel('Max Tree Depth'); axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Depth vs Performance
The overfitting problem')
axes[1].legend(fontsize=8)

plt.suptitle('Module 03 — Decision Trees', fontweight='bold')
plt.tight_layout()
plt.show()


---
## Module 04 — Random Forests

**Definition:** A Random Forest trains many decision trees independently, each on a random bootstrap sample of the data and a random subset of features at each split. Predictions are made by majority vote (classification) or averaging (regression).

**Two sources of randomness — why they help:**
- **Bagging:** Each tree trains on ~63% of data (sampled with replacement). The remaining ~37% is the Out-of-Bag (OOB) set — a free internal validation estimate, no separate val set needed.
- **Feature subsampling:** At each split, only √n_features are considered, forcing trees to specialise.

Randomness makes trees diverse. Individual errors cancel out when you average many diverse trees. This reduces *variance* without increasing *bias* — the core idea behind ensemble learning.

**Feature importance:** After training, each feature gets an importance score = how much it reduces Gini impurity across all trees and all splits. Higher = more important.

**Analogy:** Send the same claim to 300 experienced adjusters, each of whom has only seen a random subset of past claims. Individually some are wrong, but their majority vote is remarkably reliable.

**Insurance use cases:**
- Churn prediction — which customers won't renew?
- Risk scoring — combine many signals into one reliable score
- Fraud detection — captures complex non-linear patterns

---

**Scenario:** Predict which policyholders will churn at renewal. We compare an individual tree against the full forest to show the ensemble benefit.


In [ ]:
# ─── Random Forest: Churn Prediction ─────────────────────────────────────
np.random.seed(42)
n = 1500

annual_premium    = np.random.uniform(200, 2000, n)
ncb_years         = np.random.randint(0, 10, n)
nps_score         = np.random.randint(0, 11, n)
num_policies      = np.random.choice([1,2,3], n, p=[0.60, 0.30, 0.10])
had_claim         = np.random.binomial(1, 0.25, n)
prior_claims      = np.random.choice([0,1,2,3], n, p=[0.65, 0.22, 0.09, 0.04])
postcode_risk     = np.random.uniform(0, 1, n)

churn_log_odds = (
    -1.5
    + 0.8  * (annual_premium > 1200).astype(float)  # expensive policy = more likely to shop around
    + 0.6  * (nps_score < 5).astype(float)           # unhappy customer
    - 0.7  * (num_policies > 1).astype(float)         # multi-policy = more loyal
    - 0.5  * (ncb_years > 5).astype(float)            # high NCB = invested in staying
    + 0.4  * had_claim                                 # claimed this year = may move
)
will_churn = (np.random.uniform(0, 1, n) < 1/(1+np.exp(-churn_log_odds))).astype(int)

churn_df = pd.DataFrame({
    'annual_premium': annual_premium,
    'ncb_years': ncb_years,
    'nps_score': nps_score,
    'num_policies': num_policies,
    'had_claim': had_claim,
    'prior_claims': prior_claims,
    'postcode_risk': postcode_risk,
    'will_churn': will_churn
})

churn_features = ['annual_premium', 'ncb_years', 'nps_score',
                  'num_policies', 'had_claim', 'prior_claims', 'postcode_risk']
X_c = churn_df[churn_features]
y_c = churn_df['will_churn']

X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42, stratify=y_c)

print(f"Churn rate: {will_churn.mean():.1%}")
print(f"Training:   {len(X_tr):,} | Test: {len(X_te):,}")


In [ ]:
# ─── Fit and evaluate ─────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,        # 200 independent trees
    max_depth=10,
    min_samples_leaf=20,
    max_features='sqrt',     # √n_features per split = the key randomness
    class_weight='balanced',
    oob_score=True,          # free internal validation using OOB samples
    n_jobs=-1,
    random_state=42
)
rf.fit(X_tr, y_tr)

rf_proba = rf.predict_proba(X_te)[:, 1]

print(f"Test ROC-AUC:     {roc_auc_score(y_te, rf_proba):.4f}")
print(f"OOB Score:        {rf.oob_score_:.4f}  (free estimate from out-of-bag samples)")
print()

# Compare individual tree vs ensemble to show the diversity benefit
individual_aucs = [
    roc_auc_score(y_te, t.predict_proba(X_te)[:,1])
    for t in rf.estimators_[:30]
]
print(f"Avg individual tree AUC:   {np.mean(individual_aucs):.4f}")
print(f"Ensemble AUC:              {roc_auc_score(y_te, rf_proba):.4f}")
print(f"Gain from pooling:         +{roc_auc_score(y_te, rf_proba) - np.mean(individual_aucs):.4f}")


In [ ]:
# ─── Feature importance ───────────────────────────────────────────────────
# Mean decrease in Gini impurity across all trees and all splits
importances = pd.Series(rf.feature_importances_, index=churn_features).sort_values()

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['steelblue' if v >= importances.nlargest(3).min() else 'lightsteelblue'
          for v in importances.values]
ax.barh(importances.index, importances.values, color=colors, edgecolor='white')
ax.set_xlabel('Mean Decrease in Gini Impurity')
ax.set_title('Random Forest Feature Importance — Churn Prediction
Darker = top 3 features')
for i, v in enumerate(importances.values):
    ax.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()


---
## Module 05 — Gradient Boosting

**Definition:** Trees are trained *sequentially*. Each new tree is trained to predict the *residual errors* of all previous trees. The final model sums all trees' predictions, each scaled by a learning rate.

**Update rule:**  
`Fₜ(x) = Fₜ₋₁(x) + η · hₜ(x)`

Where `hₜ` is the t-th tree, and `η` (eta) is the learning rate.

**Learning rate vs number of trees tradeoff:**
- Small η (0.01–0.05) + many trees → better generalisation, slower training
- Large η (0.3+) + few trees → fast but risks overfitting

**Subsampling:** Setting `subsample < 1.0` makes each tree train on a random fraction of the data — this is *stochastic gradient boosting* and reduces overfitting.

**Key difference from Random Forest:**

| | Random Forest | Gradient Boosting |
|--|--|--|
| Trees built | In parallel (independent) | Sequentially (each corrects the last) |
| Tree depth | Deep | Shallow (weak learners) |
| Main benefit | Reduces variance | Reduces bias |

**Analogy:** A team of junior actuaries, each reviewing the previous person's work and only writing corrections. After 200 rounds, the combined output is very precise.

**Insurance use cases:**
- GLM replacement — the industry standard for pricing engines (via XGBoost/LightGBM)
- Frequency × severity pure premium modelling
- Real-time fraud scoring via API

---

**Scenario:** Predict whether a policyholder will make a claim (frequency model). We show the convergence curve — how training and test loss evolve across boosting rounds.


In [ ]:
# ─── Gradient Boosting: Claim Frequency Model ────────────────────────────
np.random.seed(42)
n = 2000

driver_age    = np.random.randint(18, 75, n)
postcode_risk = np.random.uniform(0, 1, n)
ncb_years     = np.random.randint(0, 10, n)
prior_claims  = np.random.choice([0,1,2,3], n, p=[0.65, 0.22, 0.09, 0.04])
vehicle_group = np.random.randint(1, 20, n)
avg_speed     = np.random.uniform(20, 80, n)

claim_log_odds = (
    -0.8
    + 0.5 * postcode_risk
    + 0.4 * (prior_claims > 1).astype(float)
    + 0.3 * (driver_age < 25).astype(float)
    - 0.3 * (ncb_years > 5).astype(float)
    + 0.2 * (avg_speed > 60).astype(float)
)
had_claim = (np.random.uniform(0, 1, n) < 1/(1+np.exp(-claim_log_odds))).astype(int)

gb_df = pd.DataFrame({
    'driver_age': driver_age, 'postcode_risk': postcode_risk,
    'ncb_years': ncb_years, 'prior_claims': prior_claims,
    'vehicle_group': vehicle_group, 'avg_speed': avg_speed,
    'had_claim': had_claim
})

gb_features = ['driver_age', 'postcode_risk', 'ncb_years', 'prior_claims', 'vehicle_group', 'avg_speed']
X_gb = gb_df[gb_features]
y_gb = gb_df['had_claim']

X_tr, X_te, y_tr, y_te = train_test_split(X_gb, y_gb, test_size=0.2, random_state=42)

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,   # small learning rate → more robust
    max_depth=4,          # shallow trees = weak learners
    subsample=0.8,        # stochastic boosting: 80% of data per tree
    min_samples_leaf=20,
    random_state=42
)
gb.fit(X_tr, y_tr)

gb_proba = gb.predict_proba(X_te)[:, 1]
print(f"Gradient Boosting ROC-AUC: {roc_auc_score(y_te, gb_proba):.4f}")


In [ ]:
# ─── Convergence curve (training vs test loss over boosting rounds) ────────
train_loss, test_loss = [], []

for train_pred in gb.staged_predict_proba(X_tr):
    p = np.clip(train_pred[:, 1], 1e-7, 1-1e-7)
    train_loss.append(-np.mean(y_tr * np.log(p) + (1-y_tr) * np.log(1-p)))

for test_pred in gb.staged_predict_proba(X_te):
    p = np.clip(test_pred[:, 1], 1e-7, 1-1e-7)
    test_loss.append(-np.mean(y_te * np.log(p) + (1-y_te) * np.log(1-p)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

rounds = range(1, len(train_loss)+1)
axes[0].plot(rounds, train_loss, color='darkorange', linewidth=2, label='Train Loss')
axes[0].plot(rounds, test_loss,  color='steelblue',  linewidth=2, label='Test Loss')
axes[0].set_xlabel('Boosting Rounds (trees added)'); axes[0].set_ylabel('Log Loss')
axes[0].set_title('Boosting Convergence
Ideal: both lines fall and level off together')
axes[0].legend()

# Learning rate comparison
lrs = [0.001, 0.01, 0.05, 0.1, 0.3, 0.5]
lr_aucs = []
for lr in lrs:
    gb_lr = GradientBoostingClassifier(n_estimators=100, learning_rate=lr,
                                        max_depth=4, random_state=42, subsample=0.8)
    gb_lr.fit(X_tr, y_tr)
    lr_aucs.append(roc_auc_score(y_te, gb_lr.predict_proba(X_te)[:,1]))

axes[1].plot([str(lr) for lr in lrs], lr_aucs, 'o-', color='steelblue', linewidth=2.5, markersize=8)
axes[1].set_xlabel('Learning Rate (η)'); axes[1].set_ylabel('Test ROC-AUC')
axes[1].set_title('Learning Rate vs Performance
Too high = overfit | Too low = underfit')
axes[1].grid(True, alpha=0.4)

plt.suptitle('Module 05 — Gradient Boosting', fontweight='bold')
plt.tight_layout()
plt.show()


---
## Module 06 — Support Vector Machines (SVM)

**Definition:** SVM finds the hyperplane that *maximally separates* two classes. It maximises the margin — the distance to the nearest points on each side (the support vectors). Only the support vectors determine the boundary; all other training points are irrelevant once trained.

**Objective:**  
`Maximise 2/||w||  subject to  yᵢ(w · xᵢ + b) ≥ 1`  
(maximise margin while classifying training points correctly)

**The kernel trick:** SVM can find non-linear boundaries by implicitly mapping data into higher dimensions using a *kernel function*, without ever computing the transformation explicitly.

| Kernel | Best for |
|--------|---------|
| Linear | Linearly separable data, text classification |
| RBF | Most tabular data, flexible curved boundary |
| Polynomial | Structured/image data |

**Key hyperparameters:**
- `C` — regularisation. Low C = wider margin (allows some misclassifications). High C = narrow margin (may overfit).
- `gamma` — RBF only. How far each point's influence reaches. High gamma = very local = complex boundary.

**Analogy:** Drawing a line to separate high-risk and low-risk postcodes. SVM insists on drawing the line as far as possible from any borderline case — maximising confidence. The kernel trick lets it draw curved boundaries without explicitly computing them.

**Insurance use cases:**
- Claim document classification (text features → high-dimensional sparse data)
- Anomaly detection on policy applications
- Small-data problems where tree-based methods overfit

---

**Scenario:** Compare how C affects the bias-variance tradeoff — as C increases, the model fits training data better but may overfit.


In [ ]:
# ─── SVM: Claim Classification ────────────────────────────────────────────
np.random.seed(42)
n = 1000

driver_age    = np.random.randint(18, 75, n)
postcode_risk = np.random.uniform(0, 1, n)
ncb_years     = np.random.randint(0, 10, n)
prior_claims  = np.random.choice([0,1,2,3], n, p=[0.65, 0.22, 0.09, 0.04])
vehicle_value = np.random.uniform(3000, 40000, n)

claim_log_odds = -0.8 + 0.6*postcode_risk + 0.5*(prior_claims>1) + 0.3*(driver_age<25) - 0.3*(ncb_years>5)
had_claim = (np.random.uniform(0, 1, n) < 1/(1+np.exp(-claim_log_odds))).astype(int)

svm_features = ['driver_age', 'postcode_risk', 'ncb_years', 'prior_claims', 'vehicle_value']
X_svm = pd.DataFrame({f: locals()[f] for f in ['driver_age','postcode_risk','ncb_years','prior_claims','vehicle_value']})
y_svm = pd.Series(had_claim)

X_tr, X_te, y_tr, y_te = train_test_split(X_svm, y_svm, test_size=0.2, random_state=42, stratify=y_svm)

# ─── Compare kernels ─────────────────────────────────────────────────────
print(f"{'Kernel':<10}  {'ROC-AUC':>10}  Notes")
print("-" * 52)
for kernel, note in [('linear', 'assumes linear boundary, fast'),
                     ('rbf',    'flexible — most common default'),
                     ('poly',   'polynomial boundary, degree=3')]:
    pipe_k = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel=kernel, C=1.0, class_weight='balanced',
                    probability=True, random_state=42))
    ])
    pipe_k.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, pipe_k.predict_proba(X_te)[:,1])
    print(f"  {kernel:<8}  {auc:>10.4f}  {note}")


In [ ]:
# ─── C (regularisation) effect: bias-variance tradeoff ────────────────────
C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
train_auc_c, test_auc_c = [], []

for C in C_values:
    pipe_c = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf', C=C, class_weight='balanced', probability=True, random_state=42))
    ])
    pipe_c.fit(X_tr, y_tr)
    train_auc_c.append(roc_auc_score(y_tr, pipe_c.predict_proba(X_tr)[:,1]))
    test_auc_c.append(roc_auc_score(y_te, pipe_c.predict_proba(X_te)[:,1]))

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(C_values, train_auc_c, 'o-', color='darkorange', linewidth=2, label='Train AUC')
ax.semilogx(C_values, test_auc_c,  's-', color='steelblue',  linewidth=2, label='Test AUC')
ax.set_xlabel('C (log scale)
Low C = wide margin, High C = narrow margin')
ax.set_ylabel('ROC-AUC')
ax.set_title('SVM: Effect of C on Bias-Variance Tradeoff
Where train diverges from test = overfitting begins')
ax.legend()
plt.tight_layout()
plt.show()

print("Low C  → wide margin, some misclassifications tolerated (high bias, low variance)")
print("High C → narrow margin, tries to classify all points correctly (low bias, high variance)")


---
## Module 07 — K-Nearest Neighbors (KNN)

**Definition:** KNN predicts by finding the K most similar training examples and returning their majority vote (classification) or average (regression). There is no explicit training — the entire training set becomes the model at prediction time.

**Algorithm:**
1. Receive a new policy `x`
2. Compute the Euclidean distance from `x` to every training policy
3. Find the K nearest training policies
4. Return the majority class or average value of those K neighbours

**Euclidean distance:**  
`d(x, z) = √ Σᵢ (xᵢ − zᵢ)²`

**Critical:** Features *must be standardised* before applying KNN. If vehicle value is in thousands and driver age in tens, vehicle value dominates the distance calculation entirely.

**Distance weighting:** Weight each neighbour by `1/d` — closer neighbours have more say. Almost always better than uniform weights.

**Choosing K:** Cross-validate across K = 1 to 50. K too small = overfit (noisy). K too large = oversmooth (underfits local structure).

**Analogy:** A new homeowner wants to insure their Victorian terrace. Your underwriter finds the 10 most similar historical policies and prices this one similarly.

**Insurance use cases:**
- Comparable risk pricing for specialist lines with few benchmarks
- Imputing missing premium values from similar complete records
- Cross-sell recommendations

---

**Scenario:** Predict annual premium for a new policy by finding the K most similar historical policies and averaging their premiums.


In [ ]:
# ─── KNN: Comparable Premium Pricing ─────────────────────────────────────
np.random.seed(42)
n = 1200

driver_age    = np.random.randint(18, 75, n)
ncb_years     = np.random.randint(0, 10, n)
prior_claims  = np.random.choice([0,1,2,3], n, p=[0.65, 0.22, 0.09, 0.04])
postcode_risk = np.random.uniform(0, 1, n)
vehicle_value = np.random.uniform(3000, 40000, n)

annual_premium = (
    200 + 400*postcode_risk + 200*prior_claims - 30*ncb_years
    + 0.003*vehicle_value + 100*(driver_age<25) + np.random.normal(0, 50, n)
).clip(150, 3000)

knn_df = pd.DataFrame({
    'driver_age': driver_age, 'ncb_years': ncb_years,
    'prior_claims': prior_claims, 'postcode_risk': postcode_risk,
    'vehicle_value': vehicle_value, 'annual_premium': annual_premium
})

knn_features = ['driver_age', 'ncb_years', 'prior_claims', 'postcode_risk', 'vehicle_value']
X_knn = knn_df[knn_features]
y_knn = knn_df['annual_premium']

X_tr, X_te, y_tr, y_te = train_test_split(X_knn, y_knn, test_size=0.2, random_state=42)

# CRITICAL: scale before KNN
scaler_knn   = StandardScaler()
X_tr_sc      = scaler_knn.fit_transform(X_tr)
X_te_sc      = scaler_knn.transform(X_te)

# ─── Find optimal K by MAE ────────────────────────────────────────────────
k_values  = range(1, 51)
mae_scores = []
for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k, weights='distance')
    knn.fit(X_tr_sc, y_tr)
    mae_scores.append(mean_absolute_error(y_te, knn.predict(X_te_sc)))

best_k = list(k_values)[int(np.argmin(mae_scores))]
print(f"Best K: {best_k}  |  MAE at best K: £{min(mae_scores):.2f}")
print(f"(K=1 MAE: £{mae_scores[0]:.2f}  — overfitting a single neighbour)")
print(f"(K=50 MAE: £{mae_scores[49]:.2f} — oversmoothing with 50 neighbours)")


In [ ]:
# ─── Show comparable risks for a sample new policy ────────────────────────
knn_final = KNeighborsRegressor(n_neighbors=best_k, weights='distance')
knn_final.fit(X_tr_sc, y_tr)

sample     = X_te_sc[[0]]
distances, indices = knn_final.kneighbors(sample)
predicted  = knn_final.predict(sample)[0]

print(f"New policy — predicted premium: £{predicted:.2f}")
print(f"
Top {best_k} comparable historical risks:")
comparable = X_tr.iloc[indices[0]].copy()
comparable['premium'] = y_tr.iloc[indices[0]].values
comparable['distance'] = distances[0].round(3)
print(comparable.to_string())


In [ ]:
# ─── K vs MAE plot ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_values, mae_scores, 'o-', color='steelblue', linewidth=2, markersize=4)
ax.axvline(best_k, color='crimson', linestyle='--', linewidth=2,
           label=f'Best K = {best_k}  (MAE = £{min(mae_scores):.2f})')
ax.set_xlabel('K (Number of Neighbours)'); ax.set_ylabel('Mean Absolute Error (£)')
ax.set_title('KNN: K vs MAE
Too small = overfit to noise | Too large = oversmooth')
ax.legend()
plt.tight_layout()
plt.show()


---
## Module 08 — Naive Bayes

**Definition:** Applies Bayes' Theorem to classify data. It's "naive" because it assumes all features are *conditionally independent* given the class label — a simplification that rarely holds but works well, especially for text.

**Bayes' Theorem:**  
`P(Fraud | Features) = P(Features | Fraud) · P(Fraud) / P(Features)`

- `P(Fraud)` = **Prior** — baseline fraud rate (e.g. 3%)
- `P(Features | Fraud)` = **Likelihood** — how common are these features in fraudulent claims?
- `P(Fraud | Features)` = **Posterior** — updated probability after seeing the evidence

**The naïve independence assumption:**  
`P(x₁, x₂, ..., xₙ | C) ≈ P(x₁|C) · P(x₂|C) · ... · P(xₙ|C)`  
This lets us multiply individual probabilities instead of computing joint distributions.

**Variants:**
- **Gaussian NB** — assumes each feature follows a Normal distribution within each class
- **Multinomial NB** — for count data (e.g. word frequencies in claim notes)

**Analogy:** Your fraud team knows 3% of claims are fraudulent (prior). They know 60% of fraudulent claims mention a solicitor (likelihood). When a claim mentions a solicitor, Bayes updates the probability. Add more evidence and it updates again — exactly how a detective reasons.

**Insurance use cases:**
- Claim note classification — bodily injury, property damage, theft
- Email triage — route customer correspondence to the right team
- Real-time fraud scoring — extremely fast, suitable for millisecond APIs

---

**Scenario A:** Manual Bayes' Theorem update to show the maths.  
**Scenario B:** Classify claim notes into categories using TF-IDF + Multinomial NB.


In [ ]:
# ─── Scenario A: Manual Bayes' Theorem update ────────────────────────────
np.random.seed(42)
n = 2000

prior_claims  = np.random.choice([0,1,2,3], n, p=[0.65, 0.22, 0.09, 0.04])
postcode_risk = np.random.uniform(0, 1, n)
policy_age    = np.random.randint(1, 120, n)

fraud_log_odds = -3.0 + 1.2*(prior_claims>2) + 0.8*postcode_risk + 0.6*(policy_age<6)
is_fraud = (np.random.uniform(0,1,n) < 1/(1+np.exp(-fraud_log_odds))).astype(int)

df_nb = pd.DataFrame({'prior_claims': prior_claims, 'postcode_risk': postcode_risk,
                      'policy_age': policy_age, 'is_fraud': is_fraud})

# ─── Manually compute Bayes update ───────────────────────────────────────
p_fraud_prior = is_fraud.mean()                        # P(Fraud)
p_feature_given_fraud = (df_nb[df_nb['is_fraud']==1]['prior_claims'] > 2).mean()   # P(feature | Fraud)
p_feature_given_legit = (df_nb[df_nb['is_fraud']==0]['prior_claims'] > 2).mean()   # P(feature | Legit)
p_feature             = (df_nb['prior_claims'] > 2).mean()                         # P(feature)

# Bayes: posterior = likelihood × prior / evidence
posterior = (p_feature_given_fraud * p_fraud_prior) / p_feature

print("Bayes' Theorem — manual calculation:")
print("Feature: prior_claims > 2")
print("-" * 50)
print(f"  P(Fraud)                         = {p_fraud_prior:.4f}  (baseline fraud rate)")
print(f"  P(prior_claims>2 | Fraud)        = {p_feature_given_fraud:.4f}  (likelihood)")
print(f"  P(prior_claims>2 | Legit)        = {p_feature_given_legit:.4f}")
print(f"  P(prior_claims>2)                = {p_feature:.4f}  (marginal)")
print(f"")
print(f"  → P(Fraud | prior_claims>2)      = {posterior:.4f}  ({posterior:.1%})")
print(f"")
print(f"  Prior was {p_fraud_prior:.1%}, updated to {posterior:.1%} after observing the feature.")
print(f"  That is Bayes' Theorem in action — evidence updates our belief.")


In [ ]:
# ─── Scenario B: Claim note classification with Multinomial NB ────────────
np.random.seed(42)
templates = {
    'bodily_injury': [
        "claimant reports whiplash injury after rear end collision solicitor engaged",
        "personal injury claim road traffic accident physiotherapy ongoing treatment",
        "customer sustained back neck pain soft tissue damage medical assessment",
        "injury compensation sought following collision whiplash continuing",
    ],
    'property_damage': [
        "front bumper damage low speed car park collision repair estimate provided",
        "vehicle rear end shunt minor scratches dents bodywork repair needed",
        "windscreen stone chip crack replacement glass required quote obtained",
        "side panel scrape third party at fault exchange details collected",
    ],
    'theft': [
        "vehicle stolen overnight police crime reference number provided",
        "catalytic converter stolen vehicle overnight no third party involved",
        "vehicle recovered following theft ignition damage door locks broken",
        "items stolen from vehicle smashed window laptop personal belongings",
    ]
}

notes, labels = [], []
for category, tmpl_list in templates.items():
    for _ in range(150):
        base  = np.random.choice(tmpl_list)
        extra = np.random.choice(['noted', 'confirmed', 'reported', 'verified'], 2)
        notes.append(base + ' ' + ' '.join(extra))
        labels.append(category)

notes_tr, notes_te, labels_tr, labels_te = train_test_split(
    notes, labels, test_size=0.2, random_state=42, stratify=labels
)

# TF-IDF converts text into word frequency vectors
# Multinomial NB then models the probability of each category given those frequencies
text_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=300, ngram_range=(1, 2))),
    ('nb',    MultinomialNB(alpha=0.5))   # alpha = Laplace smoothing (avoids zero probabilities)
])
text_pipe.fit(notes_tr, labels_tr)

print("Claim Note Classification:")
print(classification_report(labels_te, text_pipe.predict(notes_te)))

# Test on new examples
examples = [
    "claimant neck pain after collision solicitor now instructed",
    "front bumper damage supermarket car park low speed",
    "vehicle stolen overnight police reference number given",
]
print("New claims:")
for note, pred in zip(examples, text_pipe.predict(examples)):
    print(f"  '{note}'")
    print(f"  → {pred.upper()}")
    print()


---
## Module 09 — Neural Networks (MLP)

**Definition:** Layered sequences of mathematical transformations. Each layer applies a linear transformation followed by a non-linear activation function. This allows networks to learn arbitrarily complex patterns.

**Forward pass (one layer):**  
`a⁽ˡ⁾ = σ(W⁽ˡ⁾ · a⁽ˡ⁻¹⁾ + b⁽ˡ⁾)`

Where `W` = weight matrix, `b` = bias vector, `σ` = activation function.

**Common activation functions:**
- **ReLU** — `max(0, x)` — most common for hidden layers, avoids vanishing gradients
- **Sigmoid** — `1/(1+e⁻ˣ)` — squashes to [0,1], used for binary output
- **Softmax** — multi-class output probabilities

**Training via Backpropagation:**
1. Forward pass — compute prediction
2. Compute loss (e.g. binary cross-entropy)
3. Backward pass — compute gradient of loss w.r.t. every weight using the chain rule
4. Update weights: `w ← w − η · ∇L`

**Regularisation:**
- **Dropout** — randomly sets fraction of neurons to zero during training, forces redundancy
- **L2 / weight decay** — penalises large weights, shrinks coefficients

**Analogy:** Processing a claim photo layer by layer. Layer 1 detects edges. Layer 2 recognises parts. Layer 3 assesses damage. Each layer builds on the previous — learned automatically.

**Insurance use cases:**
- Complex pricing patterns with many non-linear feature interactions
- Fraud detection — learns subtle multi-feature combinations
- Vehicle damage assessment from photos (CNNs — same principle, different architecture)

---

**Scenario:** Train a small MLP for fraud detection. Compare it against other model families on the same task using ROC curves.


In [ ]:
# ─── Neural Network (MLP): Fraud Detection ───────────────────────────────
np.random.seed(42)
n = 1500

driver_age    = np.random.randint(18, 75, n)
ncb_years     = np.random.randint(0, 10, n)
prior_claims  = np.random.choice([0,1,2,3], n, p=[0.65, 0.22, 0.09, 0.04])
postcode_risk = np.random.uniform(0, 1, n)
policy_age    = np.random.randint(1, 120, n)
claim_amount  = np.random.lognormal(7.5, 0.8, n).clip(200, 50000)
vehicle_group = np.random.randint(1, 20, n)

fraud_log_odds = (
    -3.0
    + 1.3 * (prior_claims > 2).astype(float)
    + 0.9 * postcode_risk
    + 0.7 * (policy_age < 6).astype(float)
)
is_fraud = (np.random.uniform(0,1,n) < 1/(1+np.exp(-fraud_log_odds))).astype(int)

nn_features = ['driver_age', 'ncb_years', 'prior_claims', 'postcode_risk',
               'policy_age', 'claim_amount', 'vehicle_group']
X_nn = pd.DataFrame({f: locals()[f] for f in nn_features})
y_nn = pd.Series(is_fraud)

scaler_nn = StandardScaler()
X_nn_sc   = scaler_nn.fit_transform(X_nn)

X_tr, X_te, y_tr, y_te = train_test_split(X_nn_sc, y_nn, test_size=0.2, random_state=42, stratify=y_nn)

# ─── Architecture: 3 hidden layers (128 → 64 → 32) ───────────────────────
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),  # layer sizes (neurons per hidden layer)
    activation='relu',                  # ReLU activation
    solver='adam',                      # adaptive learning rate optimiser
    alpha=0.001,                        # L2 regularisation strength
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,               # stop if validation loss stops improving
    validation_fraction=0.1,
    random_state=42
)
mlp.fit(X_tr, y_tr)

mlp_proba = mlp.predict_proba(X_te)[:, 1]
print(f"MLP ROC-AUC:     {roc_auc_score(y_te, mlp_proba):.4f}")
print(f"Training epochs: {mlp.n_iter_}  (early stopping kicked in)")


In [ ]:
# ─── Compare all model families on the same task ──────────────────────────
from sklearn.tree import DecisionTreeClassifier

models = {
    'Logistic Regression': Pipeline([('s', StandardScaler()), ('m', LogisticRegression(class_weight='balanced', max_iter=500, random_state=42))]),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    'Neural Network (MLP)': mlp,   # already fitted above
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['darkorange', 'seagreen', 'steelblue', 'crimson', 'mediumpurple']
auc_scores = {}

for (name, model), color in zip(models.items(), colors):
    if name not in ('Neural Network (MLP)',):
        model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, proba)
    auc = roc_auc_score(y_te, proba)
    auc_scores[name] = auc
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} ({auc:.3f})')

axes[0].plot([0,1],[0,1],'k--', linewidth=1, alpha=0.4)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curves — All Model Families
Fraud Detection Task')
axes[0].legend(loc='lower right', fontsize=8)

# AUC bar chart
short_names = [n.replace(' (MLP)', '') for n in auc_scores.keys()]
axes[1].barh(short_names, list(auc_scores.values()), color=colors, edgecolor='white', alpha=0.85)
axes[1].set_xlabel('ROC-AUC'); axes[1].set_xlim(0.45, 1.0)
axes[1].axvline(0.5, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
axes[1].set_title('AUC Comparison
All models, same train/test split')
for i, (n, auc) in enumerate(auc_scores.items()):
    axes[1].text(auc + 0.002, i, f'{auc:.4f}', va='center', fontsize=9)

plt.suptitle('Module 09 — Neural Networks vs All Model Families', fontweight='bold')
plt.tight_layout()
plt.show()


---
## Module 10 — K-Means Clustering

**Definition:** Partitions data into K clusters by minimising the Within-Cluster Sum of Squares (WCSS) — total squared distance between each point and its cluster centroid.

**Algorithm:**
1. Randomly initialise K centroids
2. Assign each point to the nearest centroid
3. Recompute each centroid as the mean of its assigned points
4. Repeat steps 2–3 until assignments don't change (convergence)

**Objective:**  
`Minimise Σₖ Σₓ∈Cₖ ||x − μₖ||²`

Where `μₖ` is the centroid of cluster k.

**Choosing K:**
- **Elbow method:** Plot WCSS vs K. The "elbow" (where gains flatten) suggests the optimal K.
- **Silhouette score:** `s(i) = (b(i) − a(i)) / max(a(i), b(i))`  
  where `a(i)` = avg distance to same cluster, `b(i)` = avg distance to nearest other cluster.  
  Range: −1 to 1. Higher = better defined clusters. Choose the K that maximises this.

**Limitations:** Assumes spherical clusters. Sensitive to outliers. Cannot identify noise/anomalies (see DBSCAN for that).

**Analogy:** Tipping 500,000 policyholder files on a table and asking your team to group them into natural types without pre-defining what types to look for. K-Means does this mathematically.

**Insurance use cases:**
- Customer segmentation — natural archetypes for product design and targeted retention
- Telematics driver profiling — cluster drivers by behaviour without supervision
- Claims pattern analysis — identify clusters of similar claims to spot emerging trends


In [ ]:
# ─── K-Means: Customer Segmentation ──────────────────────────────────────
np.random.seed(42)
n = 2000

annual_premium = np.random.uniform(200, 2000, n)
ncb_years      = np.random.randint(0, 10, n)
nps_score      = np.random.randint(0, 11, n)
num_policies   = np.random.choice([1,2,3], n, p=[0.60, 0.30, 0.10])
prior_claims   = np.random.choice([0,1,2,3], n, p=[0.65, 0.22, 0.09, 0.04])
policy_months  = np.random.randint(1, 120, n)

seg_df = pd.DataFrame({
    'annual_premium': annual_premium,
    'ncb_years':      ncb_years,
    'nps_score':      nps_score,
    'num_policies':   num_policies,
    'prior_claims':   prior_claims,
    'policy_months':  policy_months
})

seg_features = list(seg_df.columns)
X_seg = StandardScaler().fit_transform(seg_df)

# ─── Elbow method + Silhouette score to find optimal K ────────────────────
inertias, silhouettes = [], []
K_range = range(2, 11)

for k in K_range:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_seg)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_seg, labels))

best_k = list(K_range)[int(np.argmax(silhouettes))]
print(f"Best K by Silhouette Score: {best_k}  (Silhouette = {max(silhouettes):.4f})")


In [ ]:
# ─── Fit and profile segments ─────────────────────────────────────────────
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
seg_df['segment'] = kmeans.fit_predict(X_seg)

profile = seg_df.groupby('segment')[seg_features].mean().round(2)
print("Segment Profiles (mean values):")
print(profile.to_string())
print()
print("Label each segment based on its profile, e.g.:")
print("  High premium + low NPS + many prior claims = 'High Risk / At Risk'")
print("  Low premium + high NCB + multi-policy      = 'Loyal Low Risk'")


In [ ]:
# ─── Visualise: Elbow, Silhouette, 2D PCA scatter ─────────────────────────
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(K_range, inertias, 'o-', color='steelblue', linewidth=2.5)
axes[0].axvline(best_k, color='crimson', linestyle='--', linewidth=1.5, label=f'Best K={best_k}')
axes[0].set_xlabel('K'); axes[0].set_ylabel('WCSS (Inertia)')
axes[0].set_title('Elbow Method
Look for the "elbow" point')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 's-', color='darkorange', linewidth=2.5)
axes[1].axvline(best_k, color='crimson', linestyle='--', linewidth=1.5, label=f'Best K={best_k}')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score
Higher = more cohesive clusters')
axes[1].legend()

pca_2d = PCA(n_components=2, random_state=42)
X_2d   = pca_2d.fit_transform(X_seg)
palette = ['steelblue', 'darkorange', 'seagreen', 'crimson', 'mediumpurple', 'grey']
for seg_id in range(best_k):
    mask = seg_df['segment'] == seg_id
    axes[2].scatter(X_2d[mask,0], X_2d[mask,1],
                    c=palette[seg_id], alpha=0.45, s=12, label=f'Seg {seg_id}')
axes[2].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
axes[2].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
axes[2].set_title('Segments in PCA Space
2D projection')
axes[2].legend(markerscale=3, fontsize=8)

plt.suptitle('Module 10 — K-Means Clustering', fontweight='bold')
plt.tight_layout()
plt.show()


---
## Module 11 — DBSCAN

**Definition:** Density-Based Spatial Clustering of Applications with Noise. Finds clusters as *dense regions* of points separated by sparse regions. Does not require K to be specified. Naturally identifies outliers/noise.

**Key concepts:**
- **Core point:** Has at least `min_samples` neighbours within radius `eps`
- **Border point:** Within `eps` of a core point but has fewer than `min_samples` neighbours of its own
- **Noise point:** Not reachable from any core point — labelled **−1** (outlier)

**Hyperparameters:**
- `eps` — neighbourhood radius. Too small = everything is noise. Too large = everything is one cluster.
- `min_samples` — minimum points to form a dense region. Rule of thumb: ≥ n_features + 1.

**Key advantages over K-Means:**
- Finds clusters of *arbitrary shape*, not just spherical
- Identifies outliers explicitly (label = −1) — very useful in insurance
- Does not require K specified in advance

**Analogy:** Your fraud investigations team overlays all claims on a map of garages and solicitors. Dense clusters appear — the same garage + same solicitor + same postcode appearing together repeatedly. DBSCAN finds those dense clusters and flags them.

**Insurance use cases:**
- Fraud ring detection — clusters of claims sharing garage, solicitor, or postcode
- Anomaly detection — individual policies that don't fit any normal group


In [ ]:
# ─── DBSCAN: Fraud Ring Detection ────────────────────────────────────────
np.random.seed(42)
n = 800

postcode_risk  = np.random.uniform(0, 1, n)
prior_claims   = np.random.choice([0,1,2,3], n, p=[0.65, 0.22, 0.09, 0.04])
claim_amount   = np.random.lognormal(7.5, 0.8, n).clip(200, 50000)
policy_age     = np.random.randint(1, 120, n)
vehicle_group  = np.random.randint(1, 20, n)

fraud_log_odds = -3.0 + 1.3*(prior_claims>2) + 0.9*postcode_risk + 0.7*(policy_age<6)
is_fraud = (np.random.uniform(0,1,n) < 1/(1+np.exp(-fraud_log_odds))).astype(int)

dbscan_df = pd.DataFrame({
    'postcode_risk': postcode_risk, 'prior_claims': prior_claims,
    'claim_amount': claim_amount, 'policy_age': policy_age,
    'vehicle_group': vehicle_group, 'is_fraud': is_fraud
})

dbscan_features = ['postcode_risk', 'prior_claims', 'claim_amount', 'policy_age', 'vehicle_group']
X_db = StandardScaler().fit_transform(dbscan_df[dbscan_features])

dbscan = DBSCAN(
    eps=0.5,           # neighbourhood radius (in standardised space)
    min_samples=5      # minimum 5 points to form a dense region
)
dbscan_df['cluster'] = dbscan.fit_predict(X_db)

n_clusters = len(set(dbscan_df['cluster'])) - (1 if -1 in dbscan_df['cluster'].values else 0)
n_noise    = (dbscan_df['cluster'] == -1).sum()

print(f"Total claims:              {n}")
print(f"Dense clusters found:      {n_clusters}  ← potential fraud rings")
print(f"Clustered claims:          {n - n_noise}")
print(f"Noise / isolated claims:   {n_noise}")
print()

# Key insight: do clustered claims have higher fraud rates?
clustered = dbscan_df[dbscan_df['cluster'] >= 0]
isolated  = dbscan_df[dbscan_df['cluster'] == -1]
print(f"Fraud rate — clustered claims:  {clustered['is_fraud'].mean():.1%}")
print(f"Fraud rate — isolated claims:   {isolated['is_fraud'].mean():.1%}")
print()
print("Clustered claims should show a higher fraud rate than isolated ones.")
print("This is the fraud ring signal DBSCAN is designed to find.")


In [ ]:
# ─── DBSCAN vs K-Means: key difference ────────────────────────────────────
X_plot = PCA(n_components=2, random_state=42).fit_transform(X_db)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# DBSCAN — noise shown in grey
labels_db = dbscan_df['cluster'].values
palette   = ['steelblue','darkorange','seagreen','crimson','mediumpurple','teal','brown']

for label in sorted(set(labels_db)):
    mask  = labels_db == label
    color = 'lightgrey' if label == -1 else palette[label % len(palette)]
    name  = 'Noise (−1)' if label == -1 else f'Cluster {label}'
    alpha = 0.3 if label == -1 else 0.7
    size  = 8  if label == -1 else 18
    axes[0].scatter(X_plot[mask,0], X_plot[mask,1], c=color, alpha=alpha, s=size, label=name if label <= 3 else '_')

axes[0].set_title(f'DBSCAN — {n_clusters} clusters
Grey = noise (no cluster), colours = dense clusters')
axes[0].legend(fontsize=8)

# K-Means — forces all points into clusters, no noise
km_labels = KMeans(n_clusters=5, random_state=42, n_init=10).fit_predict(X_db)
for k in range(5):
    mask = km_labels == k
    axes[1].scatter(X_plot[mask,0], X_plot[mask,1], c=palette[k], alpha=0.5, s=12, label=f'Cluster {k}')
axes[1].set_title('K-Means (K=5) on same data
Forces all points into clusters — cannot identify noise')
axes[1].legend(fontsize=8)

for ax in axes:
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.suptitle('Module 11 — DBSCAN vs K-Means', fontweight='bold')
plt.tight_layout()
plt.show()


---
## Module 12 — Principal Component Analysis (PCA)

**Definition:** Finds the directions of *maximum variance* in the data (principal components) and projects the data onto those axes. The first PC explains the most variance, the second explains the most remaining variance orthogonal to the first, and so on.

**Steps:**
1. Standardise features (mean=0, std=1)
2. Compute the covariance matrix
3. Find eigenvectors (directions of variance) and eigenvalues (magnitude of variance)
4. Sort by eigenvalue descending — these are your principal components
5. Project data onto the top K components

**Explained variance ratio:**  
`EVR_k = λₖ / Σᵢ λᵢ`

Where `λₖ` is the k-th eigenvalue. EVR tells you what fraction of total variance each PC captures.

**How many components to keep?** Choose the smallest K such that cumulative EVR ≥ 90% (or 95%).

**Analogy:** A telematics device captures 50+ features per trip. Most are correlated (hard braking and high speed co-occur). PCA discovers that 3 underlying "super-features" — *aggression*, *attention*, *exposure* — explain most of the variation. 50 columns → 3, without losing much information.

**Insurance use cases:**
- Telematics compression — reduce 50+ driving signals to 3–5 interpretable dimensions
- Portfolio visualisation — project your book of business into 2D for review
- Multicollinearity removal — decorrelate correlated features before regression


In [ ]:
# ─── PCA: Telematics Feature Compression ─────────────────────────────────
# Generate synthetic telematics data with 3 underlying factors
np.random.seed(42)
n = 2000

aggression = np.random.normal(0, 1, n)   # fast acceleration, hard braking
attention  = np.random.normal(0, 1, n)   # phone use, lane weaving
exposure   = np.random.normal(0, 1, n)   # night driving, motorway %

# 30 telematics signals, each a noisy combination of the 3 underlying factors
tele_data = {}
for i in range(30):
    w1, w2, w3 = np.random.uniform(-1, 1, 3)
    signal = w1*aggression + w2*attention + w3*exposure + np.random.normal(0, 0.3, n)
    tele_data[f'tele_{i+1:02d}'] = signal

tele_df = pd.DataFrame(tele_data)
X_tele  = StandardScaler().fit_transform(tele_df)

# ─── Fit PCA ──────────────────────────────────────────────────────────────
pca_full       = PCA(random_state=42)
pca_full.fit(X_tele)
cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)

n_90 = int(np.argmax(cumulative_var >= 0.90)) + 1
n_95 = int(np.argmax(cumulative_var >= 0.95)) + 1

print(f"Original features:           {X_tele.shape[1]}")
print(f"Components for 90% variance: {n_90}   (compression: {X_tele.shape[1]/n_90:.1f}× smaller)")
print(f"Components for 95% variance: {n_95}   (compression: {X_tele.shape[1]/n_95:.1f}× smaller)")

pca_3 = PCA(n_components=3, random_state=42)
X_pca = pca_3.fit_transform(X_tele)
print(f"
Top 3 PCs explain: {pca_3.explained_variance_ratio_.sum():.1%} of all variance")
for i, ev in enumerate(pca_3.explained_variance_ratio_):
    meaning = ['Aggression (speed/braking)', 'Attention (distraction)', 'Exposure (night/motorway)'][i]
    print(f"  PC{i+1}: {ev:.1%}  → likely captures '{meaning}'")


In [ ]:
# ─── Scree plot and cumulative variance ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scree plot — variance explained per component
axes[0].bar(range(1, 16), pca_full.explained_variance_ratio_[:15] * 100,
            color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot (first 15 PCs)
How much variance each PC captures')

# Cumulative variance — pick threshold
axes[1].plot(range(1, len(cumulative_var)+1), cumulative_var * 100, '-', color='steelblue', linewidth=2.5)
axes[1].axhline(90, color='darkorange', linestyle='--', linewidth=1.5, label=f'90% → {n_90} components')
axes[1].axhline(95, color='crimson',    linestyle='--', linewidth=1.5, label=f'95% → {n_95} components')
axes[1].axvline(n_90, color='darkorange', linestyle=':', linewidth=1.2)
axes[1].axvline(n_95, color='crimson',    linestyle=':', linewidth=1.2)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title('Cumulative Variance
Choose components at your threshold')
axes[1].legend()

plt.suptitle('Module 12 — PCA: Dimensionality Reduction', fontweight='bold')
plt.tight_layout()
plt.show()


---
## Module 13 — Time Series (SARIMA)

**Definition:** Time series data has one critical property — *order matters*. Tomorrow's claims depend on today's. Standard ML models ignore this. Time series methods explicitly model temporal dependencies.

**Key components:**
- **Trend** — long-term direction (portfolio grows → claim volumes rise)
- **Seasonality** — regular repeating cycles (winter weather spikes, bank holidays)
- **Autocorrelation** — current values correlate with past values
- **Noise** — random fluctuation after removing trend and seasonality

**Stationarity:** A series is stationary if its mean and variance are constant over time. Non-stationary series must be *differenced* (subtracting consecutive values) before modelling.

**SARIMA(p,d,q)(P,D,Q)ₘ:**

| Parameter | Meaning |
|-----------|---------|
| p | AR order — how many past values predict the current value |
| d | Differencing — how many times to difference to achieve stationarity |
| q | MA order — how many past residuals (errors) to include |
| P,D,Q | Seasonal equivalents of p,d,q |
| m | Seasonal period (m=12 for monthly data) |

**Analogy:** Your claims volume spikes every December (icy roads), dips in summer, and trends upward as the portfolio grows. SARIMA captures all three simultaneously and projects them forward — giving you your IBNR reserve estimate.

**Insurance use cases:**
- IBNR reserving — predict Incurred But Not Reported claims
- Premium volume forecasting — staffing and capacity planning
- Seasonal weather claims — flood/storm claim pattern modelling


In [ ]:
# ─── Time Series: Monthly Claims Forecasting ─────────────────────────────
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
months   = pd.date_range(start='2018-01-01', end='2023-12-01', freq='MS')
n_months = len(months)

# Construct a realistic series: trend + seasonality + winter bump + noise
trend       = np.linspace(800, 1200, n_months)
seasonality = 120 * np.sin(2 * np.pi * np.arange(n_months) / 12)
winter_bump = 80  * pd.DatetimeIndex(months).month.isin([11,12,1,2]).astype(float).values
noise       = np.random.normal(0, 40, n_months)

claims_ts = pd.Series(
    (trend + seasonality + winter_bump + noise).clip(400, 2000).round().astype(int),
    index=months,
    name='monthly_claims'
)

print(f"Period:  {months[0].strftime('%b %Y')} → {months[-1].strftime('%b %Y')}  ({n_months} months)")
print(f"Mean:    {claims_ts.mean():.0f} claims / month")
print(f"Range:   {claims_ts.min()} – {claims_ts.max()}")


In [ ]:
# ─── Decompose: trend + seasonality + residual ────────────────────────────
decomp = seasonal_decompose(claims_ts, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
for ax, data, title, color in zip(
    axes,
    [claims_ts, decomp.trend, decomp.seasonal, decomp.resid],
    ['Observed Monthly Claims', 'Trend Component', 'Seasonal Component (annual cycle)', 'Residual (noise)'],
    ['steelblue', 'darkorange', 'crimson', 'seagreen']
):
    ax.plot(data, color=color, linewidth=1.8)
    ax.fill_between(data.index, data, alpha=0.12, color=color)
    ax.set_ylabel('Claims')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.suptitle('Module 13 — Time Series Decomposition
Trend + Seasonality + Noise = Observed', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ─── SARIMA forecast: train on first 54 months, predict last 18 ───────────
train_ts = claims_ts[:'2022-06-01']
test_ts  = claims_ts['2022-07-01':]

# SARIMA(1,1,1)(1,1,1)_12
# p=1: one lagged value | d=1: first differencing | q=1: one lagged error
# P=1, D=1, Q=1: seasonal equivalents | 12: monthly period
sarima = SARIMAX(
    train_ts,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12)
)
sarima_fit = sarima.fit(disp=False)
forecast   = sarima_fit.get_forecast(steps=len(test_ts))
fc_mean    = forecast.predicted_mean
fc_ci      = forecast.conf_int(alpha=0.2)   # 80% confidence interval

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train_ts, color='steelblue', linewidth=2, label='Historical (training)')
ax.plot(test_ts,  color='steelblue', linewidth=2, linestyle='--', alpha=0.55, label='Actual (test)')
ax.plot(fc_mean,  color='crimson',   linewidth=2.5, label='SARIMA Forecast')
ax.fill_between(fc_ci.index, fc_ci.iloc[:,0], fc_ci.iloc[:,1],
                alpha=0.18, color='crimson', label='80% Confidence Interval')
ax.axvline(train_ts.index[-1], color='black', linestyle=':', linewidth=1.5,
           alpha=0.6, label='Train / Test split')
ax.set_ylabel('Monthly Claims')
ax.set_title('SARIMA(1,1,1)(1,1,1)₁₂  —  18-Month Claims Forecast')
ax.legend(fontsize=9)

mae_fc = mean_absolute_error(test_ts, fc_mean)
mape   = np.mean(np.abs((test_ts - fc_mean) / test_ts)) * 100
print(f"18-month forecast accuracy:")
print(f"  MAE:   {mae_fc:.1f} claims / month")
print(f"  MAPE:  {mape:.1f}%")

plt.suptitle('Module 13 — SARIMA: Claims Volume Forecasting', fontweight='bold')
plt.tight_layout()
plt.show()


---
## Module 14 — Model Evaluation & Validation

Building a model is roughly 20% of the work. Evaluating it correctly — choosing the right metrics, understanding what they mean, and ensuring the estimate generalises — is the rest.

---

### Classification Metrics

**Confusion Matrix — the foundation:**

|  | Predicted Positive | Predicted Negative |
|--|--|--|
| **Actual Positive** | TP (True Positive) | FN (False Negative — missed fraud) |
| **Actual Negative** | FP (False Positive — wrongly flagged) | TN (True Negative) |

**Derived metrics:**

`Precision = TP / (TP + FP)` — of all we flagged as fraud, how many actually were?

`Recall    = TP / (TP + FN)` — of all actual fraud, how much did we catch?

`F1 Score  = 2 · P · R / (P + R)` — harmonic mean of precision and recall

`ROC-AUC`  — area under the ROC curve (FPR vs TPR at all thresholds). Overall discrimination power.

`Gini = 2 × AUC − 1` — the standard insurance pricing performance metric.

**Which to prioritise?**
- High fraud cost → maximise Recall (catch as much fraud as possible)
- High investigation cost → maximise Precision (only flag what's likely fraud)
- F1 → balance of both

---

### Regression Metrics

`MAE  = (1/n) Σ |yᵢ − ŷᵢ|` — average absolute error, interpretable in £

`RMSE = √(1/n Σ (yᵢ − ŷᵢ)²)` — penalises large errors more heavily than MAE

`R²   = 1 − (SS_res / SS_tot)` — proportion of variance explained (0 to 1)

---

### Cross-Validation

Never evaluate on a single train/test split — results may be lucky or unlucky. K-fold CV gives a reliable estimate:
1. Split data into K folds
2. Train on K−1 folds, evaluate on the held-out fold
3. Repeat K times
4. Report mean ± std across folds


In [ ]:
# ─── Model Evaluation: Classification Metrics ─────────────────────────────
np.random.seed(42)
n = 1000

prior_claims  = np.random.choice([0,1,2,3], n, p=[0.65, 0.22, 0.09, 0.04])
postcode_risk = np.random.uniform(0, 1, n)
policy_age    = np.random.randint(1, 120, n)
claim_amount  = np.random.lognormal(7.5, 0.8, n).clip(200, 50000)
vehicle_group = np.random.randint(1, 20, n)

fraud_log_odds = -3.0 + 1.3*(prior_claims>2) + 0.9*postcode_risk + 0.7*(policy_age<6)
is_fraud = (np.random.uniform(0,1,n) < 1/(1+np.exp(-fraud_log_odds))).astype(int)

eval_features = ['prior_claims', 'postcode_risk', 'policy_age', 'claim_amount', 'vehicle_group']
X_ev = pd.DataFrame({f: locals()[f] for f in eval_features})
y_ev = pd.Series(is_fraud)

X_tr, X_te, y_tr, y_te = train_test_split(X_ev, y_ev, test_size=0.2, random_state=42, stratify=y_ev)

rf_ev = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced',
                                random_state=42, n_jobs=-1)
rf_ev.fit(X_tr, y_tr)
ev_proba = rf_ev.predict_proba(X_te)[:, 1]
ev_pred  = rf_ev.predict(X_te)

auc  = roc_auc_score(y_te, ev_proba)
gini = 2 * auc - 1

print("=" * 48)
print("Model Evaluation — Fraud Detection")
print("=" * 48)
print(f"  ROC-AUC:       {auc:.4f}")
print(f"  Gini:          {gini:.4f}  (insurance standard = 2×AUC − 1)")
print(f"  PR-AUC:        {average_precision_score(y_te, ev_proba):.4f}")
print()
print(classification_report(y_te, ev_pred, target_names=['Legitimate', 'Fraud']))


In [ ]:
# ─── ROC curve + Confusion Matrix + Score distribution ────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_te, ev_proba)
axes[0].plot(fpr, tpr, color='steelblue', linewidth=2.5, label=f'AUC = {auc:.4f}')
axes[0].plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.4, label='Random (AUC = 0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[0].set_xlabel('FPR (legitimate claims wrongly flagged)')
axes[0].set_ylabel('TPR (fraud caught)')
axes[0].set_title(f'ROC Curve
Gini = {gini:.4f}')
axes[0].legend()

# 2. Confusion Matrix
cm = confusion_matrix(y_te, ev_pred)
ConfusionMatrixDisplay(cm, display_labels=['Legitimate', 'Fraud']).plot(
    ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix')

# 3. Score distribution — separation between classes
axes[2].hist(ev_proba[y_te==0], bins=30, alpha=0.7, color='seagreen',   label='Legitimate', density=True)
axes[2].hist(ev_proba[y_te==1], bins=30, alpha=0.7, color='crimson',    label='Fraud',       density=True)
axes[2].set_xlabel('Fraud Probability Score')
axes[2].set_ylabel('Density')
axes[2].set_title('Score Distribution by Class
More separation = better model')
axes[2].legend()

plt.suptitle('Module 14 — Model Evaluation', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ─── Precision / Recall / F1 vs decision threshold ────────────────────────
# The 0.5 default threshold is rarely optimal — tune by business priority.
# High investigation cost → raise threshold (fewer flags, higher precision)
# High fraud cost         → lower threshold (more flags, higher recall)

thresh_range = np.linspace(0.01, 0.99, 100)
prec_vals, rec_vals, f1_vals = [], [], []

for t in thresh_range:
    pred = (ev_proba >= t).astype(int)
    if pred.sum() > 0:
        pv = y_te[pred==1].mean()
        rv = y_te[y_te==1].map(lambda _: pred[y_te.index[y_te==1]].mean()).mean()
        rv = pred[y_te.values==1].mean()
        prec_vals.append(pv); rec_vals.append(rv)
        f1_vals.append(2*pv*rv/(pv+rv+1e-8))
    else:
        prec_vals.append(0); rec_vals.append(0); f1_vals.append(0)

best_t_idx = int(np.argmax(f1_vals))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresh_range, prec_vals, color='darkorange', linewidth=2, label='Precision')
ax.plot(thresh_range, rec_vals,  color='steelblue',  linewidth=2, label='Recall')
ax.plot(thresh_range, f1_vals,   color='crimson',    linewidth=2, label='F1 Score')
ax.axvline(thresh_range[best_t_idx], color='black', linestyle='--', linewidth=1.5,
           label=f'Best F1 @ threshold = {thresh_range[best_t_idx]:.2f}')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Threshold
Choose threshold based on business cost of errors')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ─── Cross-validation: reliable performance estimate ─────────────────────
from sklearn.tree import DecisionTreeClassifier

print("5-Fold Stratified Cross-Validation")
print("(Each model trained and evaluated 5 times on different data splits)")
print("=" * 65)

scaler_cv = StandardScaler()
X_ev_sc   = scaler_cv.fit_transform(X_ev)

cv_models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=500),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    'Neural Network (MLP)':MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"
  {'Model':<26} {'Mean AUC':>10} {'Std':>8}  Fold-by-fold scores")
print("  " + "-" * 70)
for name, model in cv_models.items():
    scores = cross_val_score(model, X_ev_sc, y_ev, cv=cv, scoring='roc_auc', n_jobs=-1)
    fold_str = '  '.join([f'{s:.3f}' for s in scores])
    print(f"  {name:<26} {scores.mean():>10.4f} {scores.std():>8.4f}  {fold_str}")


---
## Quick Reference

### Algorithm Selection Guide

| Problem | First choice | Why |
|---------|-------------|-----|
| Premium pricing | Gradient Boosting + SHAP | Accuracy + explainability |
| Fraud detection | Random Forest / Gradient Boosting | Handles imbalance, non-linear |
| Churn prediction | Random Forest | Stable feature importance |
| Claim note classification | Multinomial NB | Fast, accurate on text |
| Customer segmentation | K-Means | Interpretable, scalable |
| Fraud ring detection | DBSCAN | Arbitrary shape, identifies noise |
| IBNR reserving | SARIMA | Trend + seasonality |
| Explainable pricing | Logistic Regression | Auditable, odds ratios |
| Feature compression | PCA | Reduces correlated signals |
| Comparable pricing | KNN | "Most similar historical risks" |

---

### Key Rules to Remember

- **Always scale features** before KNN, SVM, logistic regression, and neural networks
- **Class imbalance** → use `class_weight='balanced'` + ROC-AUC, not accuracy
- **Cross-validate** — never trust a single train/test split
- **Overfitting vs underfitting** — use learning curves to diagnose
- **Gini = 2 × AUC − 1** — the standard insurance model performance metric
- **Calibrate probabilities** — raw scores may not equal actual event probabilities
- **Interpretability vs accuracy** — tree-based models + SHAP is the industry sweet spot for regulated pricing

---

*Personal study notes | Mohammad*
